# Week 6: Single-Cell RNA-Seq Analysis

This notebook prepares the environment and data for a simplified scRNA-seq pipeline following Single-cell Best Practices (Section 3.8.2). It will:
- Build a cell-gene expression matrix (AnnData)
- Cluster cells with Leiden
- Annotate cell types with CellTypist

Deliverables:
- AnnData cell×gene matrix
- UMAP/Leiden clustering plot
- CellTypist annotation plot

Notes:
- Self-contained: installs dependencies, fetches data at runtime
- Python 3.13, no conda or pyroe
- Reference: https://www.sc-best-practices.org/introduction/raw_data_processing.html#a-real-world-example


## System Requirements Check

This section checks system and Python dependencies:
- System: `cmake`, `libtbb12`, `wget`, `curl`
- Python: `scanpy`, `anndata<0.11`, `leidenalg`, `celltypist`

Notes:
- No automatic `sudo apt-get` will be executed. Missing system packages will show a suggested command for you to run manually in a terminal.
- Python packages missing will be installed with `pip --user`.


In [ ]:
%%bash
set -u
export PATH="$HOME/.local/bin:$HOME/.cargo/bin:$PATH"

printf "\n=== System & Python Environment Check ===\n"

# --- System package checks (cmake, libtbb12, wget, curl) ---
missing_pkgs=()

check_cmd() {
  if command -v "$1" >/dev/null 2>&1; then
    printf "[OK] %s is installed at %s\n" "$1" "$(command -v "$1")"
  else
    printf "[MISSING] %s is not installed\n" "$1"
    missing_pkgs+=("$2")
  fi
}

# cmake, wget, curl are commands; libtbb12 is a Debian package
check_cmd cmake cmake
check_cmd wget wget
check_cmd curl curl

# Check libtbb12 via dpkg; fall back to checking shared object presence
if dpkg -s libtbb12 >/dev/null 2>&1; then
  echo "[OK] libtbb12 package is installed (dpkg)"
else
  if ls /usr/lib/*/libtbb.so.12 >/dev/null 2>&1; then
    echo "[OK] libtbb12 library found"
  else
    echo "[MISSING] libtbb12 (Intel TBB runtime)"
    missing_pkgs+=("libtbb12")
  fi
fi

if [ ${#missing_pkgs[@]} -gt 0 ]; then
  echo "\nMissing system packages detected: ${missing_pkgs[*]}"
  echo "Suggested install (requires sudo):"
  echo "  sudo apt-get update && sudo apt-get install -y ${missing_pkgs[*]}"
  echo "Note: This notebook will not run sudo automatically. Please run the command above in your terminal."
else
  echo "All required system packages are present."
fi

# --- Python package checks and installs (user-level) ---
python3 - <<'PY'
import sys, subprocess

print("\n--- Python packages ---")

def ensure(pkg, install_name=None, constraint=None):
    name = install_name or pkg
    spec = name if constraint is None else f"{name}{constraint}"
    try:
        mod = __import__(pkg)
        if pkg == 'anndata':
            ver = getattr(mod, '__version__', '0')
            def parse(v):
                parts = []
                for p in v.split('.'):
                    num = ''
                    for ch in p:
                        if ch.isdigit(): num += ch
                        else: break
                    parts.append(int(num or '0'))
                return tuple(parts + [0]*(3-len(parts)))
            if parse(ver) >= (0,11,0):
                print(f"[UPGRADE/DOWNGRADE] anndata {ver} -> enforcing <0.11")
                subprocess.run([sys.executable, '-m', 'pip', 'install', '--user', 'anndata<0.11'], check=False)
            else:
                print(f"[OK] anndata {ver}")
        else:
            print(f"[OK] {pkg} present")
    except Exception:
        print(f"[INSTALL] {spec}")
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--user', spec], check=False)

ensure('scanpy')
ensure('anndata', 'anndata', '<0.11')
ensure('leidenalg')
ensure('celltypist')

# Show pip location/version for clarity
subprocess.run([sys.executable, '-m', 'pip', '--version'])
PY

printf "\nEnvironment check complete. If system packages were missing, copy the suggested sudo apt-get command to your terminal and re-run this cell.\n"


## Data Download & Setup


In [ ]:
%%bash
set -u
export PATH="$HOME/.local/bin:$HOME/.cargo/bin:$PATH"

DATA_DIR="data"
BOX_URL="https://app.box.com/s/lx2xownlrhz3us8496tyu9c4dgade814"
TARBALL="toy_read_ref_set.tar.gz"
WL_GZ_URL="https://github.com/f0t1h/3M-february-2018/raw/refs/heads/master/3M-february-2018.txt.gz"
WL_GZ_PATH="$DATA_DIR/3M-february-2018.txt.gz"
WL_TXT_PATH="$DATA_DIR/3M-february-2018.txt"

mkdir -p "$DATA_DIR"

printf "\n=== Data Setup ===\n"

# 1) Always ensure whitelist first so subsequent messages aren't buried by progress
if [ -f "$WL_TXT_PATH" ]; then
  printf "Whitelist present: %s\n" "$WL_TXT_PATH"
else
  printf "Fetching 10x v3 whitelist ...\n"
  wget -q --show-progress -O "$WL_GZ_PATH" "$WL_GZ_URL" || { echo "Failed to download whitelist"; exit 1; }
  gunzip -f "$WL_GZ_PATH" || { echo "Failed to extract whitelist"; exit 1; }
  if [ -f "$WL_TXT_PATH" ]; then
    printf "Whitelist ready: %s\n" "$WL_TXT_PATH"
  else
    echo "Whitelist missing after extraction."; exit 1;
  fi
fi

# 2) Handle sample data presence / extraction
DATA_READY=0
# Canonical layout
if [ -d "$DATA_DIR/toy_ref_read/toy_read_fastq" ] && [ -d "$DATA_DIR/toy_ref_read/toy_human_ref" ]; then
  DATA_READY=1
fi

# If not canonical, normalize if top-level dirs exist
if [ "$DATA_READY" -eq 0 ] && [ -d "$DATA_DIR/toy_read_fastq" ] && [ -d "$DATA_DIR/toy_human_ref" ]; then
  printf "Found existing data at %s; normalizing layout to %s/toy_ref_read...\n" "$DATA_DIR" "$DATA_DIR"
  mkdir -p "$DATA_DIR/toy_ref_read"
  [ -d "$DATA_DIR/toy_ref_read/toy_read_fastq" ] || mv "$DATA_DIR/toy_read_fastq" "$DATA_DIR/toy_ref_read/" 2>/dev/null || true
  [ -d "$DATA_DIR/toy_ref_read/toy_human_ref" ] || mv "$DATA_DIR/toy_human_ref" "$DATA_DIR/toy_ref_read/" 2>/dev/null || true
  if [ -d "$DATA_DIR/toy_ref_read/toy_read_fastq" ] && [ -d "$DATA_DIR/toy_ref_read/toy_human_ref" ]; then
    DATA_READY=1
  fi
fi

# If still not ready, try to extract from tarball if present
if [ "$DATA_READY" -eq 0 ] && [ -f "$TARBALL" ]; then
  printf "Found %s. Extracting into %s/...\n" "$TARBALL" "$DATA_DIR"
  tar -xzf "$TARBALL" -C "$DATA_DIR" || { echo "Extraction failed"; exit 1; }
  # Normalize if tar created a top folder
  if [ -d "$DATA_DIR/toy_read_ref_set" ]; then
    [ -d "$DATA_DIR/toy_ref_read/toy_read_fastq" ] || mv "$DATA_DIR/toy_read_ref_set/toy_read_fastq" "$DATA_DIR/" 2>/dev/null || true
    [ -d "$DATA_DIR/toy_ref_read/toy_human_ref" ] || mv "$DATA_DIR/toy_read_ref_set/toy_human_ref" "$DATA_DIR/" 2>/dev/null || true
    rmdir "$DATA_DIR/toy_read_ref_set" 2>/dev/null || true
  fi
  # Ensure canonical layout under toy_ref_read
  mkdir -p "$DATA_DIR/toy_ref_read"
  [ -d "$DATA_DIR/toy_ref_read/toy_read_fastq" ] || mv "$DATA_DIR/toy_read_fastq" "$DATA_DIR/toy_ref_read/" 2>/dev/null || true
  [ -d "$DATA_DIR/toy_ref_read/toy_human_ref" ] || mv "$DATA_DIR/toy_human_ref" "$DATA_DIR/toy_ref_read/" 2>/dev/null || true

  if [ -d "$DATA_DIR/toy_ref_read/toy_read_fastq" ] && [ -d "$DATA_DIR/toy_ref_read/toy_human_ref" ]; then
    DATA_READY=1
    printf "OK: toy_ref_read directory prepared.\n"
    # Remove tarball after successful extraction to keep workspace clean
    rm -f "$TARBALL" && printf "Removed archive: %s\n" "$TARBALL" || true
  else
    echo "Warning: Expected folders not found after extraction. Contents of $DATA_DIR:"; ls -la "$DATA_DIR"
  fi
fi

# If data still missing, show Box instructions now (after whitelist progress) and exit gracefully
if [ "$DATA_READY" -eq 0 ]; then
  printf "[Action Required]\n"
  printf "Box requires a browser for download.\n"
  printf "1) Open: %s\n" "$BOX_URL"
  printf "2) Download file: %s\n" "$TARBALL"
  printf "3) Place it next to this notebook: %s/%s\n" "$(pwd)" "$TARBALL"
  printf "4) Re-run this cell.\n"
  exit 0
fi

# 3) Summary (only when data available)
printf "\nData directory summary:\n"
du -h --max-depth=2 "$DATA_DIR" 2>/dev/null || true

# List key files if present
if [ -d "$DATA_DIR/toy_ref_read/toy_human_ref/fasta" ]; then
  echo "Reference FASTA(s):"; ls -lh "$DATA_DIR/toy_ref_read/toy_human_ref/fasta" || true
fi
if [ -d "$DATA_DIR/toy_ref_read/toy_human_ref/genes" ]; then
  echo "GTF(s):"; ls -lh "$DATA_DIR/toy_ref_read/toy_human_ref/genes" || true
fi
if [ -d "$DATA_DIR/toy_ref_read/toy_read_fastq" ]; then
  echo "FASTQs:"; ls -lh "$DATA_DIR/toy_ref_read/toy_read_fastq" | head -n 20 || true
fi

## Tool Installation


In [ ]:
%%bash
# Installer for salmon, cargo, alevin-fry, simpleaf with fallbacks for simpleaf
set -u
export PATH="$HOME/.local/bin:$HOME/.cargo/bin:$PATH"
export CARGO_TERM_COLOR=always

# Set ALEVIN_FRY_HOME for simpleaf (required environment variable)
export ALEVIN_FRY_HOME="$HOME/.alevin_fry"
mkdir -p "$ALEVIN_FRY_HOME"

# Persist to .bashrc if not already present
if ! grep -q "ALEVIN_FRY_HOME" "$HOME/.bashrc" 2>/dev/null; then
  echo "export ALEVIN_FRY_HOME=\"$HOME/.alevin_fry\"" >> "$HOME/.bashrc"
fi

printf "\n=== Pipeline Tool Check/Install ===\n"
SALMON_VERSION="1.10.0"
SALMON_TARBALL="salmon-${SALMON_VERSION}_linux_x86_64.tar.gz"
SALMON_URL="https://github.com/COMBINE-lab/salmon/releases/download/v${SALMON_VERSION}/${SALMON_TARBALL}"
INSTALL_BIN="$HOME/.local/bin"
mkdir -p "$INSTALL_BIN"

status() { printf "[tool] %s\n" "$1"; }
err() { printf "[error] %s\n" "$1" >&2; }

# Pre-check for git (cargo may need it)
if ! command -v git >/dev/null 2>&1; then
  err "git not found. Install it in a terminal: sudo apt-get install -y git"
fi

# 1. cargo / Rust toolchain
if ! command -v cargo >/dev/null 2>&1; then
  status "cargo missing -> installing Rust toolchain"
  if curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y; then
    . "$HOME/.cargo/env" && status "cargo installed: $(command -v cargo)"
  else
    err "Rust installation failed. Manual: curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y"
  fi
else
  status "cargo present: $(command -v cargo)"
fi
export PATH="$HOME/.local/bin:$HOME/.cargo/bin:$PATH"

# Keep toolchain current (no sudo)
if command -v rustup >/dev/null 2>&1; then
  rustup update >/dev/null 2>&1 || true
  rustup default stable >/dev/null 2>&1 || true
fi
status "rustc: $(rustc --version 2>/dev/null || echo 'not found')"

# 2. salmon
if command -v salmon >/dev/null 2>&1; then
  status "salmon present: $(salmon --version | head -n1)"
else
  status "salmon missing -> downloading ${SALMON_TARBALL}"
  rm -f "$SALMON_TARBALL"
  if wget -q --show-progress "$SALMON_URL"; then
    if tar -xzf "$SALMON_TARBALL"; then
      if [ -f "salmon-${SALMON_VERSION}_linux_x86_64/bin/salmon" ]; then
        mv "salmon-${SALMON_VERSION}_linux_x86_64/bin/salmon" "$INSTALL_BIN/" && chmod +x "$INSTALL_BIN/salmon" && status "salmon installed"
        rm -rf "salmon-${SALMON_VERSION}_linux_x86_64" "$SALMON_TARBALL"
        status "salmon version: $(salmon --version | head -n1)"
      else
        err "Expected salmon binary not found after extraction"
      fi
    else
      err "Failed to extract $SALMON_TARBALL"
    fi
  else
    err "Download failed: $SALMON_URL"
  fi
fi

# 3. alevin-fry
if command -v alevin-fry >/dev/null 2>&1; then
  status "alevin-fry present: $(alevin-fry --version 2>&1 | head -n1)"
else
  status "alevin-fry missing -> cargo install --locked alevin-fry"
  if cargo install --locked alevin-fry; then
    status "alevin-fry installed: $(alevin-fry --version 2>&1 | head -n1)"
  else
    err "alevin-fry install failed. Install prerequisites (terminal): sudo apt-get install -y build-essential cmake pkg-config libssl-dev zlib1g-dev libcurl4-openssl-dev"
  fi
fi

# 4. simpleaf with version fallbacks (jrsonnet API break workaround)
install_simpleaf() {
  # Try latest first
  if cargo install --locked simpleaf; then
    return 0
  fi
  # Known working fallbacks (adjusted for jrsonnet changes)
  for v in 0.18.3 0.18.2 0.18.1 0.17.3; do
    status "retrying: cargo install --locked simpleaf --version ${v}"
    if cargo install --locked simpleaf --version "${v}"; then
      return 0
    fi
  done
  return 1
}

if command -v simpleaf >/dev/null 2>&1; then
  status "simpleaf present: $(simpleaf --version 2>&1 | head -n1)"
else
  status "simpleaf missing -> attempting cargo install with fallbacks"
  if install_simpleaf; then
    status "simpleaf installed: $(simpleaf --version 2>&1 | head -n1)"
  else
    err "simpleaf install failed across versions. Likely due to jrsonnet API mismatch."
    err "Options:"
    err "  1) Install build deps (terminal) then retry: sudo apt-get update && sudo apt-get install -y build-essential cmake pkg-config libssl-dev zlib1g-dev libcurl4-openssl-dev git"
    err "  2) Use alevin-fry directly (without simpleaf). I can add those cells next."
  fi
fi

# Summary
printf "\n=== Tool Summary ===\n"
for t in salmon alevin-fry simpleaf cargo; do
  if command -v "$t" >/dev/null 2>&1; then
    printf "%-12s -> %s\n" "$t" "$(command -v "$t")"
  else
    printf "%-12s -> NOT FOUND\n" "$t"
  fi
done


## Reference Indexing

In [ ]:
%%bash
set -u
export PATH="$HOME/.local/bin:$HOME/.cargo/bin:$PATH"
export ALEVIN_FRY_HOME="$HOME/.alevin_fry"

DATA_DIR="data/toy_ref_read"
REF_DIR="$DATA_DIR/toy_human_ref"
GENOME_FA="$REF_DIR/fasta/genome.fa"
GENES_GTF="$REF_DIR/genes/genes.gtf"
INDEX_DIR="data/simpleaf_index"

printf "\n=== Reference Indexing ===\n"

# Check for required files
if [ ! -f "$GENOME_FA" ]; then
  echo "Error: Reference genome not found at $GENOME_FA"
  exit 1
fi
if [ ! -f "$GENES_GTF" ]; then
  echo "Error: GTF file not found at $GENES_GTF"
  exit 1
fi

# Skip if index already exists
if [ -d "$INDEX_DIR" ] && [ -f "$INDEX_DIR/index/complete_ref_lens.bin" ]; then
  printf "Index already exists at %s\n" "$INDEX_DIR"
  printf "Index metadata:\n"
  ls -lh "$INDEX_DIR/index/" | head -n 10
  exit 0
fi

# Set paths for simpleaf
printf "Configuring simpleaf paths...\n"
simpleaf set-paths

# Build index (read length 90 for this toy dataset)
printf "\nBuilding salmon index (this may take a few minutes)...\n"
simpleaf index \
  --output "$INDEX_DIR" \
  --fasta "$GENOME_FA" \
  --gtf "$GENES_GTF" \
  --rlen 90 \
  --threads 8 \
  --no-piscem

# Verify and show results
if [ -d "$INDEX_DIR" ] && [ -f "$INDEX_DIR/index/complete_ref_lens.bin" ]; then
  printf "\n=== Index Complete ===\n"
  printf "Index location: %s\n" "$INDEX_DIR"
  printf "Index size: %s\n" "$(du -sh "$INDEX_DIR" | cut -f1)"
  printf "\nIndex contents:\n"
  ls -lh "$INDEX_DIR/index/" | head -n 10
else
  echo "Error: Index build appears to have failed"
  exit 1
fi

## Quantification

In [ ]:
%%bash
set -u
export PATH="$HOME/.local/bin:$HOME/.cargo/bin:$PATH"
export ALEVIN_FRY_HOME="$HOME/.alevin_fry"

DATA_DIR="data/toy_ref_read"
FASTQ_DIR="$DATA_DIR/toy_read_fastq"
INDEX_DIR="data/simpleaf_index"
QUANT_DIR="data/simpleaf_quant"
WHITELIST="data/3M-february-2018.txt"

printf "\n=== Quantification ===\n"

# Check prerequisites
if [ ! -d "$INDEX_DIR" ]; then
  echo "Error: Index not found at $INDEX_DIR. Run indexing cell first."
  exit 1
fi
if [ ! -f "$WHITELIST" ]; then
  echo "Error: Whitelist not found at $WHITELIST"
  exit 1
fi
if [ ! -d "$FASTQ_DIR" ]; then
  echo "Error: FASTQ directory not found at $FASTQ_DIR"
  exit 1
fi

# Skip if quantification already complete
if [ -d "$QUANT_DIR/af_quant/alevin" ] && [ -f "$QUANT_DIR/af_quant/alevin/quants_mat.mtx" ]; then
  printf "Quantification already complete at %s\n" "$QUANT_DIR"
  printf "\nMatrix dimensions:\n"
  head -n 3 "$QUANT_DIR/af_quant/alevin/quants_mat.mtx"
  printf "\nCell count: %s\n" "$(wc -l < "$QUANT_DIR/af_quant/alevin/quants_mat_rows.txt")"
  printf "Gene count: %s\n" "$(wc -l < "$QUANT_DIR/af_quant/alevin/quants_mat_cols.txt")"
  exit 0
fi

# Collect FASTQ files (R1 = barcode+UMI, R2 = cDNA)
printf "Discovering FASTQ files...\n"
R1_FILES=$(ls "$FASTQ_DIR"/*_R1_*.fastq.gz 2>/dev/null | tr '\n' ',' | sed 's/,$//')
R2_FILES=$(ls "$FASTQ_DIR"/*_R2_*.fastq.gz 2>/dev/null | tr '\n' ',' | sed 's/,$//')

if [ -z "$R1_FILES" ] || [ -z "$R2_FILES" ]; then
  echo "Error: Could not find paired R1/R2 FASTQ files in $FASTQ_DIR"
  ls -lh "$FASTQ_DIR"
  exit 1
fi

printf "R1 (barcode): %s\n" "$R1_FILES"
printf "R2 (cDNA): %s\n" "$R2_FILES"

# Run quantification
printf "\nRunning simpleaf quant (this may take several minutes)...\n"
simpleaf quant \
  --reads1 "$R1_FILES" \
  --reads2 "$R2_FILES" \
  --threads 8 \
  --index "$INDEX_DIR/index" \
  --chemistry 10xv3 \
  --resolution cr-like \
  --expected-ori fw \
  --unfiltered-pl "$WHITELIST" \
  --output "$QUANT_DIR"

# Verify and show results
if [ -f "$QUANT_DIR/af_quant/alevin/quants_mat.mtx" ]; then
  printf "\n=== Quantification Complete ===\n"
  printf "Output directory: %s\n" "$QUANT_DIR/af_quant/alevin"
  printf "Total size: %s\n" "$(du -sh "$QUANT_DIR" | cut -f1)"
  
  printf "\nMatrix Market header (genes × cells):\n"
  head -n 3 "$QUANT_DIR/af_quant/alevin/quants_mat.mtx"
  
  printf "\nDimensions:\n"
  printf "  Cells: %s\n" "$(wc -l < "$QUANT_DIR/af_quant/alevin/quants_mat_rows.txt")"
  printf "  Genes: %s\n" "$(wc -l < "$QUANT_DIR/af_quant/alevin/quants_mat_cols.txt")"
  
  printf "\nSample barcodes (first 5):\n"
  head -n 5 "$QUANT_DIR/af_quant/alevin/quants_mat_rows.txt"
  
  printf "\nSample genes (first 5):\n"
  head -n 5 "$QUANT_DIR/af_quant/alevin/quants_mat_cols.txt"
else
  echo "Error: Quantification appears to have failed"
  exit 1
fi